This notebook demonstrates a multi-GRG phenotype simulation workflow, using the sim_phenotypes_multi_grg function to simulate both continuous and binary phenotypes across multiple GRG files representing 200 samples. The simulation works by sampling causal mutations from each GRG file (1000 per file), calculating genetic values for each individual based on a normal distribution model (mean=0, variance=1), then combining the genetic values from all GRGs additively and adding environmental noise based on a specified heritability parameter (0.33 in the examples). The notebook demonstrates several scenarios including loading all GRGs into RAM simultaneously versus sequentially and generating both continuous phenotypes and binary phenotypes with a specified population prevalence (0.2). We can also optionally saving the effect sizes to parameter files for downstream analysis.

In [1]:
from grg_pheno_sim.multi_grg_phenotype import sim_phenotypes_multi_grg
from grg_pheno_sim.model import grg_causal_mutation_model


The following command only serves the purpose of converting the VCF zip file into a GRG that will be used for the phenotype simulation. The bash script below will function as expected given the relative path for the source data file is accurate.

In [ ]:
%%script bash --out /dev/null
if [ ! -f test-200-samples.grg ]; then
  grg construct -p 10 ../data/test-200-samples.vcf.gz --out-file test-200-samples.grg
fi


In [ ]:
%%script bash --out /dev/null
if [ ! -f test-200-samples_copy.grg ]; then
  grg construct -p 10 ../data/test-200-samples.vcf.gz --out-file test-200-samples_copy.grg
fi

In [14]:
%%script bash --out /dev/null
if [ ! -f test-200-samples_last.grg ]; then
  grg construct -p 10 demos/data/test-200-samples.vcf.gz --out-file test-200-samples_last.grg
fi

In [15]:
grg_list = ["test-200-samples.grg", "test-200-samples_copy.grg"]
#this is the list of GRG files to be loaded in 

We will first demonstrate loading all GRG files into RAM and simulating phenotypes. Causal mutations are sampled from each GRG, and the genetic values are obtained for the samples. The combined genetic dataframe is the addition of each GRG's genetic values. Noise is sampled at the end and added to obtain the phenotypes.

NOTE: It is necessary for each GRG to have the same number of samples.

In [16]:
model_type = "normal"
mean = 0
var = 1

model = grg_causal_mutation_model(model_type, mean=mean, var=var)

num_causal_per_file = 1000

random_seed = 1

normalize_phenotype = True #set check to True if we want phenotypes normalized

normalize_genetic_values_before_noise = False

heritability = 0.33

load_all_into_RAM = True #this parameter decides whether to load all GRGs into RAM together

save_effects = True

path_list = ['first_sample_effect_sizes.par', 'second_sample_effect_sizes.par']

multi_grg_uni_phenotypes = sim_phenotypes_multi_grg(grg_list, model, num_causal_per_file, random_seed, normalize_phenotype=normalize_phenotype, load_all_ram=load_all_into_RAM, normalize_genetic_values_before_noise=normalize_genetic_values_before_noise, heritability=heritability, save_effect_output=save_effects, effect_path_list=path_list)


In [7]:
multi_grg_uni_phenotypes

Now, we demonstrate a case where the function uses default values set for the parameters instead of custom values.

In [8]:
heritability = 0.33

load_all_into_RAM = True #this parameter decides whether to load all GRGs into RAM together

multi_grg_uni_phenotypes_default = sim_phenotypes_multi_grg(grg_list, load_all_ram=load_all_into_RAM, heritability=heritability)


In [9]:
multi_grg_uni_phenotypes_default

We now perform similar simulations, but by loading the GRGs into RAM sequentially (instead of all together).

In [10]:
model_type = "normal"
mean = 0
var = 1

model = grg_causal_mutation_model(model_type, mean=mean, var=var)

num_causal_per_file = 1000

random_seed = 1

normalize_phenotype = True #check to ensure phenotypes are normalized

normalize_genetic_values_before_noise = False

heritability = 0.33

load_all_into_RAM = False #this parameter decides whether to load all GRGs into RAM together

save_effects = True

path_list = ['first_seq_sample_effect_sizes.par', 'second_seq_sample_effect_sizes.par']

multi_grg_uni_seq_phenotypes = sim_phenotypes_multi_grg(grg_list, model, num_causal_per_file, random_seed, normalize_phenotype=normalize_phenotype, load_all_ram=load_all_into_RAM, normalize_genetic_values_before_noise=normalize_genetic_values_before_noise, heritability=heritability, save_effect_output=save_effects, effect_path_list=path_list)

In [11]:
multi_grg_uni_seq_phenotypes

Now, we demonstrate a case with binary phenotypes.

In [12]:
model_type = "normal"
mean = 0
var = 1

model = grg_causal_mutation_model(model_type, mean=mean, var=var)

num_causal_per_file = 1000

random_seed = 1

normalize_genetic_values_before_noise = False

binary=True

population_prevalence = 0.2

heritability = 0.33

load_all_into_RAM = True #this parameter decides whether to load all GRGs into RAM together

save_effects = True

path_list = ['first_seq_sample_effect_sizes.par', 'second_seq_sample_effect_sizes.par']

multi_grg_uni_bin_phenotypes = sim_phenotypes_multi_grg(grg_list, model, num_causal_per_file, random_seed, load_all_ram=load_all_into_RAM, binary=binary, population_prevalence=population_prevalence, normalize_genetic_values_before_noise=normalize_genetic_values_before_noise, heritability=heritability, save_effect_output=save_effects, effect_path_list=path_list)

In [13]:
multi_grg_uni_bin_phenotypes

In [14]:
binary_list = multi_grg_uni_bin_phenotypes["phenotype"]
num_zeros = (binary_list == 0).sum()
num_ones = (binary_list == 1).sum()

print("Number of 0s:", num_zeros)
print("Number of 1s:", num_ones)
print("Population prevalence ratio observed: ", str(num_ones/(num_ones+num_zeros)))


Finally, we demonstrate a case with more than 2 GRGs. 

In [15]:
new_grg_list = ["test-200-samples.grg", "test-200-samples_copy.grg", "test-200-samples_last.grg"]


In [16]:
model_type = "normal"
mean = 0
var = 1

model = grg_causal_mutation_model(model_type, mean=mean, var=var)

num_causal_per_file = 1000

random_seed = 1

normalize_genetic_values_before_noise = False

heritability = 0.33

load_all_into_RAM = True #this parameter decides whether to load all GRGs into RAM together

multi_grg_uni_phenotypes_three = sim_phenotypes_multi_grg(new_grg_list, model, num_causal_per_file, random_seed, load_all_ram=load_all_into_RAM, normalize_genetic_values_before_noise=normalize_genetic_values_before_noise, heritability=heritability)


In [17]:
multi_grg_uni_phenotypes_three

We can also test the standardized genotype-matrix for multi-GRG phenotype simulation. This uses the StdX operator when standardized=True and verifies that both methods loading return the expected output format.

In [17]:
num_causal_per_file = 1000
random_seed = 42
heritability = 0.33

standardized_multi_grg_ram = sim_phenotypes_multi_grg(
    grg_list,
    num_causal_per_file=num_causal_per_file,
    random_seed=random_seed,
    load_all_ram=True,
    heritability=heritability,
    standardized=True,
)

standardized_multi_grg_seq = sim_phenotypes_multi_grg(
    grg_list,
    num_causal_per_file=num_causal_per_file,
    random_seed=random_seed,
    load_all_ram=False,
    heritability=heritability,
    standardized=True,
)

required_columns = {
    "individual_id",
    "genetic_value",
    "causal_mutation_id",
    "environmental_noise",
    "phenotype",
}

assert set(standardized_multi_grg_ram.columns) == required_columns
assert set(standardized_multi_grg_seq.columns) == required_columns
assert len(standardized_multi_grg_ram) == len(standardized_multi_grg_seq)
assert standardized_multi_grg_ram.isnull().sum().sum() == 0
assert standardized_multi_grg_seq.isnull().sum().sum() == 0

standardized_multi_grg_ram.head()

For a better test, we can manually reconstruct the standardized per-GRG genetic values from the simulated causal mutations and compare them against `intermediate_genetic_vals()`. This checks that the new multi-GRG standardized path is using the `StdX` operator exactly as intended before the GRG-specific genetic values are combined.

In [19]:
import pandas as pd
import pygrgl
import numpy as np

from grg_pheno_sim.effect_size import sim_grg_causal_mutation
from grg_pheno_sim.multi_grg_phenotype import intermediate_genetic_vals
from grg_pheno_sim.ops_scipy import SciPyStdXOperator
from grg_pheno_sim.phenotype import allele_frequencies_new

model = grg_causal_mutation_model("normal", mean=0, var=1)
num_causal_per_file = 1000
random_seed = 42

manual_standardized_outputs = []
helper_standardized_outputs = []

for grg_path in grg_list:
    grg = pygrgl.load_immutable_grg(grg_path)
    causal_mutation_df = sim_grg_causal_mutation(
        grg, model=model, num_causal=num_causal_per_file, random_seed=random_seed
    )

    causal_sites = causal_mutation_df["mutation_id"].to_numpy()
    freqs = allele_frequencies_new(grg)
    beta_full = np.zeros(grg.num_mutations, dtype=float)
    beta_full[causal_sites] = causal_mutation_df["effect_size"].to_numpy()
    beta_full = beta_full.reshape(-1, 1)

    manual_genetic_values = np.squeeze(
        SciPyStdXOperator(
            grg,
            direction=pygrgl.TraversalDirection.UP,
            freqs=freqs,
            haploid=False,
        )._matmat(beta_full)
    )

    manual_df = pd.DataFrame(
        {
            "individual_id": np.arange(grg.num_individuals, dtype=int),
            "genetic_value": manual_genetic_values,
            "causal_mutation_id": 0,
        }
    ).sort_values(["individual_id", "causal_mutation_id"]).reset_index(drop=True)

    helper_df = intermediate_genetic_vals(
        grg=grg,
        model=model,
        num_causal=num_causal_per_file,
        random_seed=random_seed,
        normalize_genetic_values_before_noise=False,
        grg_name=grg_path,
        effect_path=None,
        standardized=True,
    ).sort_values(["individual_id", "causal_mutation_id"]).reset_index(drop=True)

    pd.testing.assert_frame_equal(manual_df, helper_df)

    manual_standardized_outputs.append(manual_df)
    helper_standardized_outputs.append(helper_df)

manual_combined_standardized = (
    pd.concat(manual_standardized_outputs)
    .groupby(["individual_id", "causal_mutation_id"], as_index=False)
    .agg({"genetic_value": "sum"})
    .sort_values(["individual_id", "causal_mutation_id"])
    .reset_index(drop=True)
)

helper_combined_standardized = (
    pd.concat(helper_standardized_outputs)
    .groupby(["individual_id", "causal_mutation_id"], as_index=False)
    .agg({"genetic_value": "sum"})
    .sort_values(["individual_id", "causal_mutation_id"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    manual_combined_standardized,
    helper_combined_standardized,
)

manual_combined_standardized.head()
